In [8]:
import requests
import pandas as pd
BASE_URL = "http://localhost:8000"

In [9]:
def fetch_buses(
    route_id: str,
    service_date: str,
    direction_id: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Call /iroam/buses and return two DataFrames:

    buses_df  — one row per bus:
        bus_id, trip_id, start_date, vehicle_id, n_points, n_anomalies

    events_df — one row per anomaly event:
        bus_id, trip_id, vehicle_id, event_type, minute_of_day,
        stop_index, method, approx_time_hhmm
    """
    resp = requests.get(
        f"{BASE_URL}/iroam/buses",
        params={
            "route_id": route_id,
            "service_date": service_date,
            "direction_id": direction_id,
        },
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()

    # --- buses DataFrame ---
    bus_rows = []
    for bus in data["buses"]:
        bus_rows.append({
            "bus_id":   bus["id"],
            "trip_id":     bus["trip_id"],
            "start_date":  bus["start_date"],
            "vehicle_id":  bus["vehicle_id"],
            "n_points":    len(bus["points"]),
            "n_anomalies": len(bus["anomalies"]),
        })
    buses_df = pd.DataFrame(bus_rows)

    # --- events DataFrame ---
    event_rows = []
    for bus in data["buses"]:
        for ev in bus["anomalies"]:
            hour, minute = divmod(ev["t"], 60)
            ev["approx_time_hhmm"] = f"{int(hour):02d}:{int(minute):02d}"
            event_rows.append({
                "bus_id":       bus["id"],
                "trip_id":         bus["trip_id"],
                "vehicle_id":      bus["vehicle_id"],
                "event_type":      ev["type"],
                "minute_of_day":   round(ev["t"], 2),
                "approx_time_hhmm": ev["approx_time_hhmm"],
                "stop_index":      round(ev["d"], 3),
                "method":          ev.get("method"),
                "route_id":        route_id,
                "service_date":    service_date,
                "direction_id":    direction_id,
            })
    events_df = pd.DataFrame(event_rows)

    print(f"Route {route_id} | {service_date} | direction {direction_id}")
    print(f"  {len(buses_df)} buses, {len(events_df)} anomaly events")
    if not events_df.empty:
        print(events_df["event_type"].value_counts().to_string())

    return buses_df, events_df


def fetch_bunching(route_id, service_date, direction_id, **kwargs) -> pd.DataFrame:
    """Return only bunching events."""
    _, events_df = fetch_buses(route_id, service_date, direction_id, **kwargs)
    return events_df[events_df["event_type"] == "bunch"].reset_index(drop=True)


def fetch_idle(route_id, service_date, direction_id, **kwargs) -> pd.DataFrame:
    """Return only idle events."""
    _, events_df = fetch_buses(route_id, service_date, direction_id, **kwargs)
    return events_df[events_df["event_type"] == "idle"].reset_index(drop=True)


In [ ]:
# testing
route_id = "29"
service_date = "2026-06-18"
direction_id = 0

bunching_df = fetch_bunching(route_id, service_date, direction_id)
display(bunching_df)

Route 29 | 2026-06-18 | direction 0
  76 buses, 295 anomaly events
event_type
bunch    232
crowd     39
idle      24


,bus_id,trip_id,vehicle_id,event_type,minute_of_day,approx_time_hhmm,stop_index,method,route_id,service_date,direction_id
0,2,1406020,9429,bunch,878.05,14:38,26.0,time,29,2026-06-18,0
1,2,1406020,9429,bunch,879.20,14:39,27.0,time,29,2026-06-18,0
2,2,1406020,9429,bunch,880.28,14:40,28.0,time,29,2026-06-18,0
3,2,1406020,9429,bunch,881.35,14:41,29.0,time,29,2026-06-18,0
4,2,1406020,9429,bunch,882.38,14:42,30.0,time,29,2026-06-18,0
...,...,...,...,...,...,...,...,...,...,...,...
227,68,8744020,9453,bunch,952.58,15:52,39.0,time,29,2026-06-18,0
228,68,8744020,9453,bunch,954.08,15:54,40.0,time,29,2026-06-18,0
229,73,93390020,9443,bunch,793.70,13:13,28.0,time,29,2026-06-18,0
230,73,93390020,9443,bunch,794.65,13:14,29.0,time,29,2026-06-18,0


In [ ]:
route_ids = {"29", "36"}
service_date = "2026-06-18"
direction_id = 0

# fetch 10 bunching events for each route and 10 idle events for each route; save all events to a CSV file
all_events = []
for route_id in route_ids:
    bunching_df = fetch_bunching(route_id, service_date, direction_id)
    idle_df = fetch_idle(route_id, service_date, direction_id)
    all_events.append(bunching_df.head(10))
    all_events.append(idle_df.head(10))
all_events_df = pd.concat(all_events, ignore_index=True)
all_events_df.to_csv("groundtruth_events.csv", index=False)

# save csv file to out/qa/groundtruth_events.csv
#all_events_df.to_csv("/Users/devrajsolanki/Documents/iROAM/out/qa/groundtruth_events.csv", index=False)

Route 36 | 2026-06-18 | direction 0
  79 buses, 252 anomaly events
event_type
bunch    225
idle      27
Route 36 | 2026-06-18 | direction 0
  79 buses, 252 anomaly events
event_type
bunch    225
idle      27
Route 29 | 2026-06-18 | direction 0
  76 buses, 295 anomaly events
event_type
bunch    232
crowd     39
idle      24
Route 29 | 2026-06-18 | direction 0
  76 buses, 295 anomaly events
event_type
bunch    232
crowd     39
idle      24
